In [1]:
import os
import re
from collections import defaultdict, Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import polars as pl
import loguru as logger

from settings.settings_analyze_efizz import Settings_ae
from behave_analysis.process.session import get_experiment
from behave_analysis.utils.rayleigh.load_rayleigh import collect_all_rayleigh_paths, load_all_rayleigh_data
from behave_analysis.utils.rayleigh.manipulate_rayleigh_df import extract_compartment_values, extract_firing_rates
from behave_analysis.utils.creating_directories import make_directory
from settings.settings_analyze_efizz import Settings_ae as Settings
from behave_analysis.analyze.TunED.model import TunEdModel

from behave_analysis.database.Experiments.JAL003_ex import JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept
from behave_analysis.database.Experiments.JAL004_ex import JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept
from behave_analysis.database.Experiments.JAL005_ex import JAL005_8thSept, JAL005_21stSept
from behave_analysis.database.Experiments.JAL006_ex import JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_flip3_18mar, JAL6_flip7_1apr
from behave_analysis.database.Experiments.JAL007_ex import JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr, JAL7_30apr
from behave_analysis.database.Experiments.JAL008_ex import JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_tiny_3may, JAL8_flip4_10may, JAL8_14may, JAL8_21may

In [64]:
experiments_objects = [JAL6_flip7_1apr, JAL6_flip3_18mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_28mar,
                       JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept,
                       JAL005_8thSept, JAL005_21stSept,
                       JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr,
                       JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_flip4_10may, JAL8_14may,
                       JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept]

tinny_barrier = [JAL8_tiny_3may, JAL8_21may, JAL7_30apr]

# Mice groups based on session names
mice_groups = {
    "JAL6": ['JAL6_flip7_1apr', 'JAL6_flip3_18mar', 'JAL6_flip4_21mar', 'JAL6_flip5_25mar', 'JAL6_28mar'],
    "JAL3": ['JAL3_25aug', 'JAL3_1sept', 'JAL3_4sept', 'JAL3_7sept'],
    "JAL7": ['JAL7_sesh8_9apr', 'JAL7_sesh9_16apr', 'JAL7_flip5_22mar', 'JAL7_flip2_12mar', 'JAL7_23apr'],
    "JAL8": ['JAL8_flip1_25apr', 'JAL8_flip2_29apr', 'JAL8_flip4_10may', 'JAL8_14may'],
    "JAL4": ['JAL4_3rdSept', 'JAL4_19thSept', 'JAL4_28aug', 'JAL4_11thSept'],
    "JAL5": ['JAL5_8thSept', 'JAL5_21stSept']}


session_NAMES = ["JAL6_flip7_1apr", "JAL6_flip3_18mar", "JAL6_flip4_21mar", "JAL6_flip5_25mar", "JAL6_28mar",
                 "JAL3_25aug", "JAL3_1sept", "JAL3_4sept", "JAL3_7sept", 
                 "JAL005_8thSept", "JAL005_21stSept",
                 "JAL7_sesh8_9apr", "JAL7_sesh9_16apr", "JAL7_flip5_22mar", "JAL7_flip2_12mar", "JAL7_23apr",
                 "JAL8_flip1_25apr", "JAL8_flip2_29apr", "JAL8_flip4_10may", "JAL8_14may",
                 "JAL4_3rdSept", "JAL4_19thSept", "JAL4_28aug", "JAL4_11thSept"]

In [3]:
def regex(angle):
    "Remove unwanted characters from angle file string"
    pattern = r'^(.*?)(?=_Rayleigh)'
    match = re.search(pattern, angle)
    assert match, f"Could not find match for {angle}"
    return match.group(0)

def nest_dic():
    """A function to create arbitrarily nested dictionaries"""
    return defaultdict(nest_dic)

# Init Params
conditions = ["shelter_only", "barrier_pre_flip", "barrier_post_flip"]
total_cells = 0
total_sessions = 0
rayleigh_threshold = 0.15
fr_threshold = 5 # Hz
dir = make_directory(r"Z:\Jasmine_Laurence\rayleigh_analysis")
angle_keys = ['hdir_Rayleigh.arrow', 'hsa_Rayleigh.arrow', 'h_postflipbar_a_Rayleigh.arrow', 'h_preflipbar_a_Rayleigh.arrow']

# TODO firing rate threshold not implemented

threat_dict = nest_dic() # dict[session][cell][condition] = {max_rayleigh_angle}
cell_count = 0
not_meet_threshold = 0

for i, session in enumerate(experiments_objects):
    loaded_session = get_experiment(session)
    paths = collect_all_rayleigh_paths(session = loaded_session, cluster_type = "good", conditions= conditions) # paths[condition][angles]
    condition_data = load_all_rayleigh_data(paths) # condition_data[condition][angles] ~ hdir, hsa, center, post flip, pre flip
    
    # if condition_data keys is empty, skip
    if len(condition_data["shelter_only"].keys()) == 0:
        print(f"Skipping session {session} as no data found")
        continue

    
    # Count the number of sessions and cells
    print(condition_data["shelter_only"].keys())
    nCells = len(condition_data["shelter_only"]["hdir_Rayleigh.arrow"]["Rayleigh"]) # Cells are the same for all conditions and angles in one session
    cell_count += nCells
    
    for cell in range(nCells):
        tuned = False # A pointer to check if the cell is tuned to any angle in any condition

        for ci, condition in enumerate(condition_data.keys()):
            rayleigh = 0 # Per condition reset the rayleigh value
            for angle in angle_keys:
                
                #Just the threat zone
                output = extract_compartment_values(condition_data[condition][angle], column_name="Rayleigh") # Extract the rayleigh values for THREAT ONLY
                if np.logical_and(output[cell][1] > rayleigh, output[cell][1] > rayleigh_threshold):
                    rayleigh = output[cell][1]
                    max_angle_str = regex(angle)
                    tuned = True
                                        
            # For each cell in each session, save the max rayleigh values and corresponding condition | angle combo
            try:
                if tuned:
                    threat_dict[i][cell][condition] = max_angle_str
                else:
                    threat_dict[i][cell][condition] = "Not tuned"
            except:
                print(f"Cell {cell} in session {i} in {condition} did not meet threshold")
                
        if not tuned:
            # Track the number of cells that did not meet the threshold
            not_meet_threshold += 1

dict_keys(['hsa_Rayleigh.arrow', 'hdir_Rayleigh.arrow', 'h_bar_centre_a_Rayleigh.arrow', 'h_postflipbar_a_Rayleigh.arrow', 'h_preflipbar_a_Rayleigh.arrow'])
Skipping session Experiment(nick_name='JAL006', total_sessions=9, mouse_number_pyrat='BAA-1104292', experiment_file_names=None, root_path=WindowsPath('JAL006'), experiment_name='flip', experiment_idx=3, experiment_date='2024_03_18', experiment_time='11_53_29', experiment_path=WindowsPath('JAL006_barrier_flip2_2024_03_18T11_53_29'), shelter_time=[0.25, -1], barrier_time=[68.25, -1], barrier_flip_time=180.25) as no data found
Skipping session Experiment(nick_name='JAL006', total_sessions=9, mouse_number_pyrat='BAA-1104292', experiment_file_names=None, root_path=WindowsPath('JAL006'), experiment_name='flip', experiment_idx=4, experiment_date='2024_03_21', experiment_time='11_20_34', experiment_path=WindowsPath('JAL006_shelter_barrier_flip_3_2024_03_21T11_20_34'), shelter_time=[0.25, -1], barrier_time=[59.25, -1], barrier_flip_time=172

In [4]:
# Counts across sessions
TC_shelter = []
TC_barrier_pre_flip = []
TC_barrier_post_flip = []
for session in threat_dict:
    for cell in threat_dict[session]:
        x = threat_dict[session][cell]["shelter_only"]
        TC_shelter.append(x)
        y = threat_dict[session][cell]["barrier_pre_flip"]
        TC_barrier_pre_flip.append(y)
        z = threat_dict[session][cell]["barrier_post_flip"]
        TC_barrier_post_flip.append(z)

TC_shelter_counts = Counter(TC_shelter)
TC_barrier_pre_flip_counts = Counter(TC_barrier_pre_flip)
TC_barrier_post_flip_counts = Counter(TC_barrier_post_flip)
print("Across session counts")
print(TC_shelter_counts)
print(TC_barrier_pre_flip_counts)

TC_shelter_total = 0
TC_barrier_pre_flip_total = 0
TC_barrier_post_flip_total = 0
c1 = sum([i for i in TC_shelter_counts.values()])
c2 = sum([i for i in TC_barrier_pre_flip_counts.values()])
c3 = sum([i for i in TC_barrier_post_flip_counts.values()])
print(f"Cells in shelter_only: {c1}")
print(f"Cells in barrier_pre_flip: {c2}")
print(f"Cells in barrier_post_flip: {c3}")
print(f"Total cells: {cell_count}")

# Counts within sessions
TC_within_session_counts_shelter = defaultdict(lambda: defaultdict(int))
TC_within_session_counts_bar_pre_flip = defaultdict(lambda: defaultdict(int))
TC_within_session_counts_bar_post_flip = defaultdict(lambda: defaultdict(int))

for session in threat_dict:
    for cell in threat_dict[session]:
        x = threat_dict[session][cell]["shelter_only"]
        y = threat_dict[session][cell]["barrier_pre_flip"]
        z = threat_dict[session][cell]["barrier_post_flip"]
        TC_within_session_counts_shelter[session][x] += 1
        TC_within_session_counts_bar_pre_flip[session][y] += 1
        TC_within_session_counts_bar_post_flip[session][z] += 1

Across session counts
Counter({'Not tuned': 2009, 'h_preflipbar_a': 645, 'h_postflipbar_a': 621, 'hdir': 545, 'hsa': 488})
Counter({'Not tuned': 1731, 'h_preflipbar_a': 795, 'hdir': 741, 'h_postflipbar_a': 564, 'hsa': 477})
Cells in shelter_only: 4308
Cells in barrier_pre_flip: 4308
Cells in barrier_post_flip: 4308
Total cells: 4308


In [5]:
# create two subplots that share the same y axis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
xcoords = [0, 1, 3, 4]

# THREAT --------------------------------------------------------------------------------------------

# Plot the threat zone only
ax1.bar(xcoords, 
        [TC_barrier_pre_flip_counts['h_preflipbar_a'] / cell_count, 
         TC_barrier_pre_flip_counts['h_postflipbar_a'] / cell_count, 
         TC_barrier_post_flip_counts['h_preflipbar_a'] / cell_count, 
         TC_barrier_post_flip_counts['h_postflipbar_a'] / cell_count],
        color= 'darkorchid',
        alpha = 0.7
        )
ax1.set_xticks(xcoords, 
           ['Open Edge | Pre flip condition', 
            'Closed Edge | Pre flip condition', 
            'Closed Edge | Post flip condition', 
            'Open Edge | Post flip condition'],
           rotation=10)
ax1.set_ylabel('Fraction of cells', fontsize=16)

# Counts within sessions
TC_within_session_counts_shelter = defaultdict(lambda: defaultdict(int))
TC_within_session_counts_bar_pre_flip = defaultdict(lambda: defaultdict(int))
TC_within_session_counts_bar_post_flip = defaultdict(lambda: defaultdict(int))

TC_mouse_dict = defaultdict(list)

for session, session_name in zip(threat_dict, session_names):
    
    # Find the mouse for the current session
    mouse = None
    for key, sessions in mice_groups.items():
        if session_name in sessions:
            mouse = key
            break
    
    for cell in threat_dict[session]:
        x = threat_dict[session][cell]["shelter_only"]
        y = threat_dict[session][cell]["barrier_pre_flip"]
        z = threat_dict[session][cell]["barrier_post_flip"]
        TC_within_session_counts_shelter[session][x] += 1
        TC_within_session_counts_bar_pre_flip[session][y] += 1
        TC_within_session_counts_bar_post_flip[session][z] += 1
    
    # Scatter plot points
    y_scatter_values = [
        TC_within_session_counts_bar_pre_flip[session]['h_preflipbar_a'] / sum(TC_within_session_counts_bar_pre_flip[session].values()),
        TC_within_session_counts_bar_pre_flip[session]['h_postflipbar_a'] / sum(TC_within_session_counts_bar_pre_flip[session].values()),
        TC_within_session_counts_bar_post_flip[session]['h_preflipbar_a'] / sum(TC_within_session_counts_bar_post_flip[session].values()),
        TC_within_session_counts_bar_post_flip[session]['h_postflipbar_a'] / sum(TC_within_session_counts_bar_post_flip[session].values())
    ]
        
    ax1.scatter(xcoords, y_scatter_values, color='darkorchid', s=100, alpha=1, marker="")
    
    # Draw lines between scatter points for each session
    ax1.plot(xcoords[:2], y_scatter_values[:2], color='darkorchid', alpha=0.5, linewidth = .75)  # Connect points at 0 and 1
    ax1.plot(xcoords[2:], y_scatter_values[2:], color='darkorchid', alpha=0.5, linewidth = .75)  # Connect points at 3 and 4
    
    TC_mouse_dict[mouse].append(y_scatter_values)

# Apply average mouse information
for mouse in TC_mouse_dict:
    yvals = np.mean(np.array(TC_mouse_dict[mouse]), axis=0)
    ax1.plot(xcoords[:2], yvals[:2], linestyle='-', color="darkorchid", linewidth=3, label=f'{mouse} Average')
    ax1.plot(xcoords[2:], yvals[2:], linestyle='-', color="darkorchid", linewidth=3)
    
plt.show()

# Store the cell ids in a session -> cluster id -> top two rayleigh values + other thresholds ready for the tuned model

In [80]:
def regex(angle):
    "Remove unwanted characters from angle file string"
    pattern = r'^(.*?)(?=_Rayleigh)'
    match = re.search(pattern, angle)
    assert match, f"Could not find match for {angle}"
    return match.group(0)

def nest_dic():
    """A function to create arbitrarily nested dictionaries"""
    return defaultdict(nest_dic)

# Init Params
conditions = ["shelter_only", "barrier_pre_flip", "barrier_post_flip"]
total_cells = 0
total_sessions = 0
rayleigh_threshold = 0.15
fr_threshold = 5 # Hz
dir = make_directory(r"Z:\Jasmine_Laurence\rayleigh_analysis")
angle_keys = ['hdir_Rayleigh.arrow', 'hsa_Rayleigh.arrow', 'h_postflipbar_a_Rayleigh.arrow', 'h_preflipbar_a_Rayleigh.arrow']

# TODO firing rate threshold not implemented

threat_dict = nest_dic() # dict[session][cell][condition] = {max_rayleigh_angle}
cell_count = 0
not_meet_threshold = 0

import pandas as pd

for i, session in enumerate(experiments_objects):
    loaded_session = get_experiment(session)
    paths = collect_all_rayleigh_paths(session = loaded_session, cluster_type = "good", conditions= conditions) # paths[condition][angles]
    condition_data = load_all_rayleigh_data(paths) # condition_data[condition][angles] ~ hdir, hsa, center, post flip, pre flip
    try:
        cluster_msater = pd.read_csv(os.path.join(loaded_session.base_path, loaded_session.processed_path) + "\\" + "spike_count_by_frame_and_goodcluster.csv")
    except:
        print(f"Could not load cluster master for session {session}")
        continue
    
    # extract the spike_clusters column and sort them uniquely from smallest
    spike_clusters = sorted(cluster_msater['spike_clusters'].unique())
    
    
    # if condition_data keys is empty, skip
    if len(condition_data["shelter_only"].keys()) == 0:
        print(f"Skipping session {session} as no data found")
        continue

    
    # Count the number of sessions and cells
    print(condition_data["shelter_only"].keys())
    nCells = len(condition_data["shelter_only"]["hdir_Rayleigh.arrow"]["Rayleigh"]) # Cells are the same for all conditions and angles in one session
    assert nCells == len(spike_clusters), f"Number of cells {nCells} does not match number of clusters {len(spike_clusters)}"
    cell_count += nCells
    
    for cell in range(nCells):
        tuned = False # A pointer to check if the cell is tuned to any angle in any condition

        for ci, condition in enumerate(condition_data.keys()):
            rayleigh = 0 # Per condition reset the rayleigh value
            for angle in angle_keys:
                
                #Just the threat zone
                output = extract_compartment_values(condition_data[condition][angle], column_name="Rayleigh") # Extract the rayleigh values for THREAT ONLY
                if np.logical_and(output[cell][1] > rayleigh, output[cell][1] > rayleigh_threshold):
                    rayleigh = output[cell][1]
                    max_angle_str = regex(angle)
                    tuned = True
                                        
            # For each cell in each session, save the max rayleigh values and corresponding condition | angle combo
            try:
                if tuned:
                    threat_dict[session_NAMES[i]][spike_clusters[cell]][condition] = max_angle_str
                else:
                    threat_dict[session_NAMES[i]][spike_clusters[cell]][condition] = "Not tuned"
            except:
                print(f"Cell {cell} in session {i} in {condition} did not meet threshold")
                
        if not tuned:
            # Track the number of cells that did not meet the threshold
            not_meet_threshold += 1

dict_keys(['hsa_Rayleigh.arrow', 'hdir_Rayleigh.arrow', 'h_bar_centre_a_Rayleigh.arrow', 'h_postflipbar_a_Rayleigh.arrow', 'h_preflipbar_a_Rayleigh.arrow'])
Could not load cluster master for session Experiment(nick_name='JAL006', total_sessions=9, mouse_number_pyrat='BAA-1104292', experiment_file_names=None, root_path=WindowsPath('JAL006'), experiment_name='flip', experiment_idx=3, experiment_date='2024_03_18', experiment_time='11_53_29', experiment_path=WindowsPath('JAL006_barrier_flip2_2024_03_18T11_53_29'), shelter_time=[0.25, -1], barrier_time=[68.25, -1], barrier_flip_time=180.25)
Could not load cluster master for session Experiment(nick_name='JAL006', total_sessions=9, mouse_number_pyrat='BAA-1104292', experiment_file_names=None, root_path=WindowsPath('JAL006'), experiment_name='flip', experiment_idx=4, experiment_date='2024_03_21', experiment_time='11_20_34', experiment_path=WindowsPath('JAL006_shelter_barrier_flip_3_2024_03_21T11_20_34'), shelter_time=[0.25, -1], barrier_time=[

In [82]:
print(threat_dict.keys())
print(threat_dict["JAL6_flip7_1apr"].keys())
print(threat_dict["JAL6_flip7_1apr"][5])
print(threat_dict["JAL6_flip7_1apr"][6])
print(threat_dict["JAL6_flip7_1apr"][7])


dict_keys(['JAL6_flip7_1apr', 'JAL6_flip5_25mar', 'JAL6_28mar', 'JAL3_25aug', 'JAL3_1sept', 'JAL3_4sept', 'JAL3_7sept', 'JAL005_8thSept', 'JAL005_21stSept', 'JAL7_sesh8_9apr', 'JAL7_sesh9_16apr', 'JAL7_flip5_22mar', 'JAL7_flip2_12mar', 'JAL7_23apr', 'JAL8_flip1_25apr', 'JAL8_flip2_29apr', 'JAL8_flip4_10may', 'JAL8_14may', 'JAL4_3rdSept', 'JAL4_19thSept', 'JAL4_28aug', 'JAL4_11thSept'])
dict_keys([5, 6, 7, 8, 17, 18, 19, 20, 21, 22, 25, 26, 27, 28, 29, 32, 33, 35, 37, 38, 39, 40, 41, 43, 44, 45, 46, 47, 48, 52, 53, 54, 56, 58, 59, 60, 61, 62, 64, 65, 73, 74, 75, 81, 82, 83, 84, 90, 92, 93, 97, 98, 100, 101, 102, 103, 104, 107, 111, 114, 115, 118, 121, 122, 128, 129, 132, 134, 135, 140, 141, 142, 145, 146, 149, 150, 151, 154, 156, 157, 162, 163, 166, 172, 173, 176, 179, 181, 182, 185, 187, 188, 189, 192, 199, 200, 202, 204, 205, 208, 212, 217, 219, 220, 223, 224, 228, 229, 236, 240, 249, 250, 251, 253, 255, 258, 259, 260, 261, 263, 264, 265, 266, 267, 268, 270, 271, 274, 277, 280, 281, 2

to do:
- get the actual values of the rayleigh
- return the second highest angle